![neural_net](../images/neural_network_2_3_2_1.svg)

In [18]:
import numpy as np

X = np.array([
    [200, 15],
    [210, 14],
    [190, 16],
    [205, 13],
    [150, 20],
    [250, 5],
    [260, 8],
    [120, 25],
    [195, 12],
    [170, 22],
    [215, 17],
    [185, 18],
    [230, 10],
    [140, 24],
    [180, 13],
    [220, 12],
    [165, 19],
    [245, 6],
    [198, 17],
    [155, 21],
])  # shape (20, 2)

y = np.array([1,1,1,1,0,0,0,0,1,0,1,1,0,0,1,1,0,0,1,0])  # shape (20,)

In [19]:
"""
Data normalization using z-score normalization (standardization). 
This method transforms the data to have a mean of 0 and a standard deviation of 1.
"""
X_normalized = (X - np.mean(X, axis=0)) / np.std(X, axis=0)

print(X_normalized.shape)
print(X_normalized)

(20, 2)
[[ 0.16072839 -0.06366004]
 [ 0.43547777 -0.24554588]
 [-0.11402099  0.11822579]
 [ 0.29810308 -0.42743172]
 [-1.21301852  0.84576915]
 [ 1.5344753  -1.88251842]
 [ 1.80922468 -1.33686091]
 [-2.03726667  1.75519834]
 [ 0.0233537  -0.60931756]
 [-0.66351976  1.20954082]
 [ 0.57285246  0.30011163]
 [-0.25139568  0.48199747]
 [ 0.98497653 -0.97308923]
 [-1.4877679   1.5733125 ]
 [-0.38877038 -0.42743172]
 [ 0.71022715 -0.60931756]
 [-0.80089445  0.66388331]
 [ 1.39710061 -1.70063258]
 [ 0.10577851  0.30011163]
 [-1.07564383  1.02765498]]


<h2 style="text-align: center;">Weighted sum</h2>

The linear score is calculated as the weighted sum of features plus a bias term aka Linear Regression, for multiple features you would have multiple weighted sums:

$$z = f_{w,b}(x) = w_1 x_1 + w_2 x_2 + b$$

Where:
* $w_1, w_2$ represent the weights (parameters)
* $x_1$ represents the temperature
* $x_2$ represents the duration
* $b$ represents the bias (intercept)


<h2 style="text-align: center;">Sigmoid Function Formula</h2>

The function that maps any real-valued number into a strict range between 0 and 1:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Where:
* $z$ is the input score from the linear model
* $e$ is Euler's number (~2.71828)

In this case for the neural network this is our activation formula.
Basically $\sigma(z)$ is our output of each neuron, which gets fed into every other neuron at the next layer.

In [20]:
"""
    Z = w1x + w2y + b
    This is before the activation function is applied. 
    The output of this equation is called the weighted sum or linear combination of inputs. 
    The weights (w1, w2) and bias (b) are parameters that the model learns during training,
    in order to minimize the error between predicted and actual outputs.
"""


def linear_combination(X, weights, bias):
    return np.dot(X, weights) + bias



"""
    This is the actual activation function that is applied to the weighted sum.
    The activation function introduces non-linearity to the model, allowing it to learn complex patterns in the data. 
    Common activation functions include sigmoid, tanh, and ReLU (Rectified Linear Unit). 
    The choice of activation function can significantly impact the performance of the neural network.
"""

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [21]:
"""
    Cost function (loss function) is used to measure the difference between the predicted output and the actual output.
    The goal of training a neural network is to minimize this cost function.
    We do this by adjusting the weights and biases of the model using optimization algorithms like gradient descent.
    
    The function below is the cost function for binary classification problems, known as binary cross-entropy loss or log loss.
    
    Where:
    - y_true is the true label (0 or 1)
    - y_pred is the predicted probability of the positive class (1)
    - y_true * np.log(y_pred) is the contribution to the loss when the true label is 1
    - (1 - y_true) * np.log(1 - y_pred) is the contribution to the loss when the true label is 0
    - The negative sign is used to ensure that the loss is always positive, as we want to minimize it.
"""

def binary_cross_entropy_loss(y_true, y_pred):
    m = y_true.shape[0]
    loss = -(1/m) * np.sum(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
    return loss



"""
    This is the gradient descent algorithm, which is used to optimize the weights and biases of the model.
    The algorithm works by iteratively updating the weights and biases in the direction of the negative gradient
    of the cost function.
    
    Done in iterations not epochs
    
"""

def gradient_descent(X, y, weights, bias, learning_rate, iterations):
    m = X.shape[0]
    for i in range(iterations):
        # Forward pass
        z = linear_combination(X, weights, bias)
        y_pred = sigmoid(z)

        # Compute the loss
        loss = binary_cross_entropy_loss(y, y_pred)

        # Compute gradients
        dz = y_pred - y
        dw = (1/m) * np.dot(X.T, dz)
        db = (1/m) * np.sum(dz)

        # Update weights and bias
        weights -= learning_rate * dw
        bias -= learning_rate * db

        # Print loss every 10 iterations
        if i % 10 == 0:
            print(f"Iteration {i}: Loss = {loss}")

    return weights, bias

In [ ]:
"""
    This is the actual layer and neuron functions that will be used to build the neural network.
    By combining the linear combination and activation function, we can create a single neuron that takes in inputs,
    applies weights and bias, and produces an output.
    For a multi-layer neural network, we can stack multiple layers of neurons on top of each other,
    where the output of one layer becomes the input to the next layer.
    Every neuron in a layer is connected to every neuron in the next layer, forming a fully connected network.
    
    For this particular example we will be using a neural network with the following architceture:
    Input layer: 2 neurons (for the two features in the dataset)
    Hidden layer 1: 3 neurons (with sigmoid activation function)
    Hidden layer 2: 2 neurons (with sigmoid activation function)
    Output layer: 1 neuron (with sigmoid activation function for binary classification)
    
    Where dense layer is a layer of neurons that are fully connected to the previous layer.
    
"""

def neuron(X, weights, bias):
    z = linear_combination(X, weights, bias)
    return sigmoid(z)
